# Kauã Guimarães Esperança Moreira
# RA: 241024196
# TP - 1

In [168]:
# importando bibliotecas e carregando os dados dos datasets, no formato latino por conta dos 
# caracteres especiais e pulando a primeira linha, que não  faz parte dos dados
import pandas as pd
educacao = pd.read_csv("Education.csv", encoding='latin1', skiprows=1)
internet = pd.read_csv("InternetUsage.csv", encoding='latin1', skiprows=1)

In [169]:
# Visualizando as primeiras linhas de cada tabela para entender como os dados estão organizados 
# e identificar quais colunas serão relevantes para a análise
educacao.head()

,Region/Country/Area,Unnamed: 1,Year,Series,Value,Footnotes,Source
0,1,"Total, all countries or areas",2005,Students enrolled in primary education (thousa...,"679,272",NaN,"United Nations Educational, Scientific and Cul..."
1,1,"Total, all countries or areas",2005,Gross enrollment ratio - Primary (male),105.2,NaN,"United Nations Educational, Scientific and Cul..."
2,1,"Total, all countries or areas",2005,Gross enrollment ratio - Primary (female),100.4,NaN,"United Nations Educational, Scientific and Cul..."
3,1,"Total, all countries or areas",2005,Students enrolled in lower secondary education...,"309,713",NaN,"United Nations Educational, Scientific and Cul..."
4,1,"Total, all countries or areas",2005,Gross enrollment ratio - Lower secondary level...,81.0,NaN,"United Nations Educational, Scientific and Cul..."


In [170]:
# como estão os dados da tabela internet
internet.head()

,Region/Country/Area,Unnamed: 1,Year,Series,Value,Footnotes,Source
0,1,"Total, all countries or areas",2005,Percentage of individuals using the internet,15.6,NaN,"International Telecommunication Union (ITU), G..."
1,1,"Total, all countries or areas",2010,Percentage of individuals using the internet,28.4,NaN,"International Telecommunication Union (ITU), G..."
2,1,"Total, all countries or areas",2015,Percentage of individuals using the internet,39.8,NaN,"International Telecommunication Union (ITU), G..."
3,1,"Total, all countries or areas",2020,Percentage of individuals using the internet,58.6,NaN,"International Telecommunication Union (ITU), G..."
4,1,"Total, all countries or areas",2021,Percentage of individuals using the internet,61.7,NaN,"International Telecommunication Union (ITU), G..."


In [171]:
# renomeando as colunas
educacao.columns = ["ID", "Região", "Ano", "Série", "Taxa de matrícula(%)", "Rodapé", "Fonte"]
internet.columns = ["ID", "Região", "Ano", "Tema", "Porcentagem de indivíduos com acesso à internet(%)", "Rodapé", "Fonte"]

In [172]:
# Verificando a existência de valores nulos em cada coluna, para entender se seria necessário 
# tratar ou remover dados incompletos.
print("Dados faltantes na tabela da educação:\n",educacao.isnull().sum())
print("\nDados faltantes na tabela do acesso à internet:\n",internet.isnull().sum())
# Foi identificada a presença de uma coluna de rodapé, contendo indicações como “estimate”. 
# Como essa variável não impacta diretamente a análise, ela foi ignorada. 
# Os valores estimados foram mantidos, pois ainda representam informações válidas

Dados faltantes na tabela da educação:
 ID                         0
Região                     0
Ano                        0
Série                      0
Taxa de matrícula(%)       0
Rodapé                  5862
Fonte                      0
dtype: int64

Dados faltantes na tabela do acesso à internet:
 ID                                                      0
Região                                                  0
Ano                                                     0
Tema                                                    0
Porcentagem de indivíduos com acesso à internet(%)      0
Rodapé                                                300
Fonte                                                   0
dtype: int64


In [173]:
# Renomeando e limpando colunas inúteis
educacao = educacao[["Região", "Ano", "Série", "Taxa de matrícula(%)"]]
internet = internet[["Região", "Ano", "Porcentagem de indivíduos com acesso à internet(%)"]]

# removendo as linhas que não possuem esses dados principais
educacao = educacao.dropna(subset=["Taxa de matrícula(%)"])
internet = internet.dropna(subset=["Porcentagem de indivíduos com acesso à internet(%)"])

In [174]:
#transforma os valores que podem estar em string para numerico
educacao["Taxa de matrícula(%)"] = educacao["Taxa de matrícula(%)"].astype(str).str.replace(",", "").astype(float)
internet["Porcentagem de indivíduos com acesso à internet(%)"] = internet["Porcentagem de indivíduos com acesso à internet(%)"].astype(str).str.replace(",", "").astype(float)

educacao["Ano"] = educacao["Ano"].astype(int)
internet["Ano"] = internet["Ano"].astype(int)

In [175]:
#separa somente as taxas, para podermos ter uma base que faça sentido na comparação, 
# pois pegando a quantidade absoluta, dependeria do tamanho do país também, 
# isso influenciaria na análise
educacao = educacao[educacao["Série"].str.contains("Gross enrollment ratio", na=False)]

In [176]:
# Defini os anos que seriam analisados, escolhendo pontos específicos para comparar a 
# evolução ao longo do tempo.
anos = [2005, 2015, 2023]

educacao = educacao[educacao["Ano"].isin(anos)]
internet = internet[internet["Ano"].isin(anos)]

In [177]:
#Extraí informações da coluna "Série" para criar novas variáveis: nível escolar e gênero, 
# facilitando a análise posteriormente.
educacao["Nível escolar"] = educacao["Série"].str.extract(r'(Primary|Lower secondary level|Upper secondary level)')
educacao["Gênero"] = educacao["Série"].str.extract(r'(male|female)')
educacao = educacao.drop(columns=["Série"])

In [178]:
# Realizei a união dos dois datasets utilizando as colunas "Região" e "Ano", 
# mantendo apenas os registros que existem em ambas as tabelas
dados = pd.merge(educacao, internet, on=["Região", "Ano"], how="inner")

In [179]:
dados.head()

,Região,Ano,Taxa de matrícula(%),Nível escolar,Gênero,Porcentagem de indivíduos com acesso à internet(%)
0,"Total, all countries or areas",2005,105.2,Primary,male,15.6
1,"Total, all countries or areas",2005,100.4,Primary,female,15.6
2,"Total, all countries or areas",2005,81.0,Lower secondary level,male,15.6
3,"Total, all countries or areas",2005,76.9,Lower secondary level,female,15.6
4,"Total, all countries or areas",2005,52.0,Upper secondary level,male,15.6


In [180]:
dados.dtypes

Região                                                 object
Ano                                                     int64
Taxa de matrícula(%)                                  float64
Nível escolar                                          object
Gênero                                                 object
Porcentagem de indivíduos com acesso à internet(%)    float64
dtype: object